In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, DynamicCache
from transformers.cache_utils import Cache
import torch as t
import torch.nn as nn
import torch.nn.functional as F
from typing import Annotated

TARGET_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
DRAFT_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda"


# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
  TARGET_MODEL_ID,
  use_fast=True,
)

# load models
target_model = AutoModelForCausalLM.from_pretrained(
  TARGET_MODEL_ID,
  dtype=t.bfloat16,
  device_map=DEVICE,
)

draft_model = AutoModelForCausalLM.from_pretrained(
  DRAFT_MODEL_ID,
  dtype=t.bfloat16,
  device_map=DEVICE,
)

print(target_model.config)

# init cache
target_cache = DynamicCache(config=target_model.config)
draft_cache = DynamicCache(config=draft_model.config)


target_model.eval()
draft_model.eval()

In [31]:
def generate(message: str, model: nn.Module, max_tokens: int = 200, cache: Cache = None):
  messages = [{"role": "user", "content": message}]
  input = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
  ).to(draft_model.device)

  input_ids = input.input_ids
  
  for _ in range(max_tokens):
    with t.no_grad():
      if cache:
        logits = model(input_ids=input_ids[:, cache.get_seq_length():], past_key_values=cache).logits
      else:
        logits = model(input_ids=input_ids).logits
    next_token_logits = logits[0, -1, :]
    next_token = t.argmax(next_token_logits, keepdim=True)
    if next_token.item() == tokenizer.eos_token_id:
      break
    input_ids = t.cat([input_ids, next_token.unsqueeze(0)], dim=-1)
    print(tokenizer.decode(next_token), end="") 

In [3]:
def draft_loop(input_ids: Annotated[t.Tensor, "B S"], draft_length: int, temperature: float, greedy: bool = False, cache: Cache = None):
  batch_size, seq_len = input_ids.shape
  vocab_size = draft_model.config.vocab_size

  sample_ids = t.empty(batch_size, draft_length, device=draft_model.device, dtype=input_ids.dtype)
  draft_probs = t.empty(batch_size, draft_length, vocab_size, device=draft_model.device)
  
  for i in range(draft_length):
    with t.no_grad():
      if cache:
        input_tokens = input_ids if i == 0 else sample_ids[:, i-1:i]
        logits = draft_model(input_ids=input_tokens, past_key_values=cache).logits
      else:
        input_tokens = t.cat([input_ids,  sample_ids[:, :i]], dim=-1)
        logits = draft_model(input_ids=input_tokens).logits
    next_token_logits = logits[0, -1, :]
    next_token_probs = F.softmax(next_token_logits.float() /temperature, dim=-1)
    draft_probs[0, i] = next_token_probs
    if greedy:
      next_token = t.argmax(next_token_logits, dim=-1)
    else:
      next_token = t.multinomial(next_token_probs, num_samples=1)
    sample_ids[0, i] = next_token

  return sample_ids, draft_probs


# verify loop
# do forward pass
def verify_loop(
  input_ids: Annotated[t.Tensor, "B S"],
  sample_ids: Annotated[t.Tensor, "B S"],
  draft_probs: Annotated[t.Tensor, "B S V"],
  draft_length: int,
  temperature: float,
  greedy: bool = False,
  cache: Cache = None
) -> Annotated[t.Tensor, "B L"]:
  with t.no_grad():
    if cache:
      block = t.cat([input_ids[:, cache.get_seq_length():], sample_ids], dim=-1)
      logits = target_model(input_ids=block, past_key_values=cache).logits
    else:
      logits = target_model(input_ids=t.cat([input_ids,  sample_ids], dim=-1)).logits
      
  target_logits = logits[:, -(draft_length+1):, :]
  target_probs = F.softmax(target_logits.float() /temperature, dim=-1)

  confirmed_tokens = t.empty(1, draft_length+1, device=target_model.device, dtype=int)
  include_bonus = True
  for i in range(draft_length):
    sample_id = sample_ids[0, i]

    p_x = target_probs[0, i, sample_id]
    q_x = draft_probs[0, i, sample_id]

    acceptance_rate = min(1., p_x.item()/q_x.item())
    accept = (sample_id == target_probs[0, i].argmax()) if greedy else t.rand(1).item() < acceptance_rate

    if accept: # accept X. add to input ids
      confirmed_tokens[0, i] = sample_id 
    else:
      p, q = target_probs[0, i], draft_probs[0, i]
      residual = t.clamp(p - q, min=0)
      residual = residual / residual.sum()
      if greedy:
        next_token = t.argmax(target_probs[0, i], dim=-1)
      else:
        next_token = t.multinomial(residual, num_samples=1)
      confirmed_tokens[0, i] = next_token
      confirmed_tokens = confirmed_tokens[:, :i+1]
      include_bonus = False
      break
  if include_bonus:
    if greedy:
      bonus = t.argmax(target_probs[0, draft_length], dim=-1)
    else:
      bonus = t.multinomial(target_probs[0, draft_length], num_samples=1)
    confirmed_tokens[0, draft_length] = bonus
  
  return confirmed_tokens

In [25]:
draft_length = 5

def speculative_decoding(
  message: str,
  max_tokens: int = 200,
  temperature: float = 1.0,
  greedy: bool = False,
  target_cache: Cache = None,
  draft_cache: Cache = None
) -> list[int]:
  messages = [{"role": "user", "content": message}]
  input = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
  ).to(draft_model.device)

  # draft loop
  input_ids = input.input_ids
  valid = True
  generated_tokens = 0
  accepted_draft_token_counts = []
  while valid and generated_tokens < max_tokens:
    sample_ids, draft_probs = draft_loop(input_ids, draft_length, temperature, greedy=greedy, cache=draft_cache)
    confirmed_tokens = verify_loop(input_ids, sample_ids, draft_probs, draft_length, temperature, greedy=greedy, cache=target_cache)
    # rollback draft cache
    accepted = confirmed_tokens.shape[-1] - 1
    if target_cache:
      target_cache.crop(input_ids.shape[1] + accepted)
    if draft_cache:
      draft_cache.crop(input_ids.shape[1] + accepted)
    
    generated_tokens += confirmed_tokens.shape[-1]
    accepted_draft_token_counts.append(confirmed_tokens.shape[-1]-1)
    
    for i in range(confirmed_tokens.shape[-1]):
      token_id = confirmed_tokens[0, i]
      if token_id.item() == tokenizer.eos_token_id:
        valid = False
        break
      print(tokenizer.decode(token_id), end="")

    input_ids = t.cat([input_ids,  confirmed_tokens], dim=-1)
  
  return accepted_draft_token_counts

In [22]:
from contextlib import contextmanager
import time

@contextmanager
def cuda_timer(label=""):
  t.cuda.synchronize()
  start = time.perf_counter()
  yield
  t.cuda.synchronize()
  print(f"{label}: {time.perf_counter() - start:.3f}s")

In [34]:
target_cache = DynamicCache(config=target_model.config)

with cuda_timer("generate"):
  generate("Recite the English alphabet, then count to 50", target_model, cache=None)
  print("")


Certainly! Here is the English alphabet:

A, B, C, D, E, F, G, H, I, J, K, L, M, N, O, P, Q, R, S, T, U, V, W, X, Y, Z

And here is the count to 50:

1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
generate: 7.951s


In [ ]:
target_cache = DynamicCache(config=target_model.config)
draft_cache  = DynamicCache(config=draft_model.config)

with cuda_timer("spec"):
  accepted_counts = speculative_decoding("Recite the English alphabet, then count to 50", temperature=1.0, greedy=False, target_cache=target_cache, draft_cache=draft_cache)
  print("")


import numpy as np
print(f"acceptance rate:{np.sum(accepted_counts)/(len(accepted_counts)*5):.2f}. avg accepted count: {np.mean(accepted_counts):.2f}")

Certainly! Here is the English alphabet:

A, B, C, D, E, F, G, H, I, J, K, L, M, N, O, P, Q, R, S, T, U, V, W, X, Y, Z

And here is the count to 50:

1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 
spec: 1.829s
acceptance rate:0.91. avg accepted count: 4.54


In [16]:
target_cache.get_seq_length()

241

## KV Cache

In [8]:
class KVCache:
  def __init__(self, max_cache_size: int, n_kv_heads: int, head_dim: int, device=None, dtype=None):
    self.device = device
    self.dtype = dtype

    self._k_cache = t.empty(n_kv_heads, max_cache_size, head_dim, dtype=dtype, device=device)
    self._v_cache = t.empty_like(self._k_cache)
    self._ptr = 0

  def reset(self):
    self._ptr = 0

  def rollback(self, n_rejected: int):
    self._ptr -= n_rejected
  
  def update(self, k_new: Annotated[t.Tensor, "n_kv_heads n_new head_dim"], v_new: Annotated[t.Tensor, "n_kv_heads n_new head_dim"]) -> tuple[t.Tensor, t.Tensor]:
    n_new = k_new.shape[0]

    assert self._ptr + n_new <= self._k_cache.shape[0], "cache full"

    self._k_cache[:, self._ptr:self._ptr+n_new] = k_new
    self._v_cache[:, self._ptr:self._ptr+n_new] = v_new

    self._ptr += n_new

    return self._k_cache[:, :self._ptr], self._v_cache[:, :self._ptr]